<a href="https://colab.research.google.com/github/fourmodern/2025_aidrugdiscovery/blob/main/20260824/notebooks/protein_design_rfdiffusion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RFdiffusion + ProteinMPNN de novo 단백질 설계 실습

> **대상**: 신약개발/단백질공학 재직자·전공자 · **환경**: Google Colab

## ⚠️ 실행 전 필수 확인
- **GPU 런타임 필수**: `런타임 → 런타임 유형 변경 → 하드웨어 가속기 → GPU (T4 이상)`. RFdiffusion 은 CPU 로는 사실상 실행 불가.
- **설치·가중치 다운로드에 수 분**: RFdiffusion/ColabDesign 설치와 모델 체크포인트(및 AlphaFold 파라미터) 다운로드로 첫 셀은 수 분~10분 이상 걸릴 수 있습니다.
- **이 노트북은 GPU 없는 로컬에서 문법만 검증**되었습니다. 설치 명령·CLI 인자는 아래 공식 레시피를 그대로 따랐으나, **최종 동작은 Colab GPU 런타임에서 직접 확인**하세요.

## 근거로 삼은 공식 레시피 (install/실행 명령의 출처)
- **설치**: sokrypton/ColabDesign 공식 노트북 `rf/examples/diffusion.ipynb` 의 setup 셀 — <https://github.com/sokrypton/ColabDesign/blob/main/rf/examples/diffusion.ipynb>
- **RFdiffusion 실행 CLI**: RosettaCommons/RFdiffusion README (unconditional monomer) — <https://github.com/RosettaCommons/RFdiffusion>
- **ProteinMPNN 실행 CLI**: dauparas/ProteinMPNN README (`protein_mpnn_run.py`) — <https://github.com/dauparas/ProteinMPNN>


## 개요 — 두 모델의 역할과 설계 흐름

de novo(무에서) 단백질 설계는 보통 **두 단계**로 나뉩니다.

| 단계 | 도구 | 하는 일 | 관점 |
|------|------|---------|------|
| ① 백본 생성 | **RFdiffusion** | 원자 좌표(3D 백본 구조)를 **확산 모델(diffusion)** 로 생성. 서열은 아직 없음 | 구조를 먼저 만든다 |
| ② 서열 설계 | **ProteinMPNN** | 주어진 백본 구조에 **접혀서 그 모양이 되는 아미노산 서열**을 설계 (inverse folding) | 구조에 맞는 서열을 채운다 |
| ③ (선택) 검증 | **ESMFold** 등 | 설계 서열을 다시 접어(fold) 원 백본과 얼마나 일치하는지 확인 (self-consistency) | 서열↔구조 왕복 검증 |

**전체 흐름**: `RFdiffusion(백본) → ProteinMPNN(서열) → (선택) ESMFold 로 재접힘 → 원 백본과 비교`

**신약·단백질공학 맥락**: 새로운 결합 단백질(binder)·효소·스캐폴드를 데이터베이스의 자연 서열에 의존하지 않고 목적에 맞게 처음부터 설계할 수 있어, 항체 대체 바인더·미니단백질 치료제·바이오센서 등에 활용됩니다. 다만 **생성물은 컴퓨터가 낸 가설**이며 실제 발현·안정성·기능은 실험으로 검증해야 합니다(마지막 섹션 참조).

이 노트북은 가장 단순한 예인 **unconditional monomer(제약 없는 단량체) 100 residue 백본 생성**으로 전체 파이프라인을 한 번 통과시킵니다.


## 0. GPU 확인 + 설치

먼저 GPU 가 붙어 있는지 확인합니다.

In [ ]:
# GPU 확인 (없으면 런타임 유형을 GPU 로 변경 후 재실행)
!nvidia-smi -L
!nvidia-smi --query-gpu=name,memory.total --format=csv 2>/dev/null || echo "GPU 미할당 — 런타임 유형을 GPU 로 변경하세요"


In [ ]:
#@title RFdiffusion + ColabDesign 설치 (약 3~10분, GPU 런타임 필수)
# 아래 셀은 sokrypton/ColabDesign 공식 노트북 rf/examples/diffusion.ipynb 의 setup 셀을
# 그대로 옮긴 것입니다 (추측으로 지어낸 명령 아님).
#   출처: https://github.com/sokrypton/ColabDesign/blob/main/rf/examples/diffusion.ipynb
import os, time, sys

if not os.path.isdir("params"):
    os.system("apt-get install aria2 -y")
    os.system("mkdir params")
    # 파라미터 다운로드를 백그라운드로 시작 (RFdiffusion 체크포인트 + AlphaFold 파라미터)
    os.system("(\
aria2c -q -x 16 https://files.ipd.uw.edu/krypton/schedules.zip; \
aria2c -q -x 16 http://files.ipd.uw.edu/pub/RFdiffusion/6f5902ac237024bdd0c176cb93063dc4/Base_ckpt.pt; \
aria2c -q -x 16 http://files.ipd.uw.edu/pub/RFdiffusion/e29311f6f1bf1af907f9ef9f44b8328b/Complex_base_ckpt.pt; \
aria2c -q -x 16 http://files.ipd.uw.edu/pub/RFdiffusion/f572d396fae9206628714fb2ce00f72e/Complex_beta_ckpt.pt; \
aria2c -q -x 16 https://storage.googleapis.com/alphafold/alphafold_params_2022-12-06.tar; \
tar -xf alphafold_params_2022-12-06.tar -C params; \
touch params/done.txt) &")

if not os.path.isdir("RFdiffusion"):
    print("installing RFdiffusion...")
    os.system("git clone https://github.com/sokrypton/RFdiffusion.git")
    os.system("pip install jedi omegaconf hydra-core icecream pyrsistent pynvml decorator")
    os.system("pip install git+https://github.com/NVIDIA/dllogger#egg=dllogger")
    os.system("pip install --no-dependencies dgl -f https://data.dgl.ai/wheels/torch-2.4/cu124/repo.html")
    os.system("pip install --no-dependencies e3nn==0.5.5 opt_einsum_fx")
    os.system("cd RFdiffusion/env/SE3Transformer; pip install .")
    os.system("wget -qnc https://files.ipd.uw.edu/krypton/ananas")
    os.system("chmod +x ananas")

if not os.path.isdir("colabdesign"):
    print("installing ColabDesign...")
    os.system("pip -q install git+https://github.com/sokrypton/ColabDesign.git")
    os.system("ln -s /usr/local/lib/python3.*/dist-packages/colabdesign colabdesign")

if not os.path.isdir("RFdiffusion/models"):
    print("downloading RFdiffusion params (백그라운드 다운로드 완료 대기)...")
    os.system("mkdir RFdiffusion/models")
    models = ["Base_ckpt.pt", "Complex_base_ckpt.pt", "Complex_beta_ckpt.pt"]
    for m in models:
        while os.path.isfile(f"{m}.aria2"):
            time.sleep(5)
    os.system(f"mv {' '.join(models)} RFdiffusion/models")
    os.system("unzip schedules.zip; rm schedules.zip")

if "RFdiffusion" not in sys.path:
    os.environ["DGLBACKEND"] = "pytorch"
    sys.path.append("RFdiffusion")

print("설치 완료 (또는 이미 설치됨).")


### 0-1. Colab Python 3.13 호환 패치 (dgl)

현재 Colab은 **Python 3.13**입니다. RFdiffusion이 쓰는 **구버전 dgl**은 `from collections import Mapping`
(파이썬 3.10+에서 `collections.abc`로 이동)을 사용해 import 오류가 납니다. 아래 셀이 dgl 소스를 자동 수정합니다.
(설치 셀 실행 직후 **한 번** 실행하세요.)

In [ ]:
# dgl를 Python 3.13 호환으로 패치 — 'from collections import Mapping,...' → 'from collections.abc import ...'
import importlib.util, os, re, glob
spec = importlib.util.find_spec("dgl")            # import 하지 않고 위치만 확인
assert spec and spec.origin, "dgl 미설치 — 위 설치 셀을 먼저 실행하세요."
dgl_dir = os.path.dirname(spec.origin)
NAMES = "Mapping|MutableMapping|Iterable|Iterator|Sequence|Callable|Set|Hashable|Container"
pat = re.compile(rf"from collections import ((?:{NAMES})(?:\s*,\s*(?:{NAMES}))*)")
n=0
for f in glob.glob(dgl_dir+"/**/*.py", recursive=True):
    try: s=open(f, encoding="utf-8").read()
    except Exception: continue
    s2=pat.sub(r"from collections.abc import \1", s)
    if s2!=s:
        open(f,"w",encoding="utf-8").write(s2); n+=1
print(f"dgl Python 3.13 호환 패치: {n}개 파일 수정 @ {dgl_dir}")
# 검증: 이제 import 가능해야 함
import dgl; print("dgl import OK:", dgl.__version__)


In [ ]:
# 시각화용 py3Dmol 설치 (경량)
!pip -q install py3Dmol
import requests  # ESMFold API 호출용 (Colab 기본 포함)
print("py3Dmol / requests 준비 완료")


## 1. RFdiffusion — unconditional monomer 백본 생성

가장 단순한 예: **아무 제약 없이 길이 100 residue 의 백본을 생성**합니다.

- `contigmap.contigs=[100-100]` : 100 residue 단일 사슬을 새로 생성하라는 의미
- `inference.num_designs` : 만들 백본 개수
- `diffuser.T` : 확산 스텝 수 (기본 50)

> 근거(RosettaCommons/RFdiffusion README, unconditional monomer 예시):
> `./scripts/run_inference.py 'contigmap.contigs=[150-150]' inference.output_prefix=test_outputs/test inference.num_designs=10`
> ColabDesign clone 환경에서는 스크립트 경로가 `./RFdiffusion/run_inference.py` 입니다.


In [ ]:
import os
os.makedirs("outputs", exist_ok=True)

LENGTH = 100        # 생성할 백본 길이 (residue 수)
NUM_DESIGNS = 2     # 생성할 백본 개수 (1~2개면 실습에 충분)
DIFFUSER_T = 50     # 확산 스텝 수 (기본 50)

# RFdiffusion 실행 — hydra CLI. 출처: RosettaCommons/RFdiffusion README
!DGLBACKEND=pytorch ./RFdiffusion/run_inference.py 'contigmap.contigs=[{LENGTH}-{LENGTH}]' inference.output_prefix=outputs/monomer inference.num_designs={NUM_DESIGNS} diffuser.T={DIFFUSER_T}


In [ ]:
import glob
# RFdiffusion 산출물: outputs/monomer_0.pdb, outputs/monomer_1.pdb ...
pdbs = sorted(glob.glob("outputs/monomer_*.pdb"))
print("생성된 백본 PDB 목록:", pdbs)
backbone_pdb = pdbs[0] if pdbs else "outputs/monomer_0.pdb"
print("이후 단계에서 사용할 첫 번째 백본:", backbone_pdb)


In [ ]:
# py3Dmol 로 생성된 백본 시각화 (서열 없음 = 구조만)
import py3Dmol
with open(backbone_pdb) as f:
    pdb_data = f.read()

view = py3Dmol.view(width=600, height=450)
view.addModel(pdb_data, "pdb")
view.setStyle({"cartoon": {"color": "spectrum"}})
view.zoomTo()
view.show()
print("RFdiffusion 이 생성한 백본 (아직 아미노산 서열은 정해지지 않은 상태)")


### 1-1. [강화] 생성된 여러 백본 비교 (다중 모델 그리드)

RFdiffusion 은 확률적이라 실행마다 다른 백본이 나옵니다. 아래는 `outputs/monomer_*.pdb` 로 **실제로 생성된 백본들**을
py3Dmol 그리드에 나란히 놓고, 각 사슬을 **잔기 index N→C 그라디언트(spectrum, 파랑=N말단 → 빨강=C말단)** 로 칠해
전체 fold 를 한눈에 비교합니다. (파일이 없으면 안내만 출력합니다.)

In [ ]:
# [강화] 생성된 여러 백본을 py3Dmol 그리드로 비교 (잔기 index 그라디언트 = spectrum)
import os, glob
import py3Dmol

bb_pdbs = sorted(glob.glob("outputs/monomer_*.pdb"))
if not bb_pdbs:
    print("백본 PDB(outputs/monomer_*.pdb)가 없습니다 — 1단계(RFdiffusion) 실행 후 다시 시도하세요.")
else:
    n = len(bb_pdbs)
    cols = min(n, 2)
    rows = (n + cols - 1) // cols
    view = py3Dmol.view(width=340 * cols, height=300 * rows, viewergrid=(rows, cols))
    for k, p in enumerate(bb_pdbs):
        r, c = divmod(k, cols)
        with open(p) as fh:
            view.addModel(fh.read(), "pdb", viewer=(r, c))
        # 잔기 index N->C 그라디언트 (spectrum) — 단량체라 체인은 A 하나
        view.setStyle({"model": 0}, {"cartoon": {"color": "spectrum"}}, viewer=(r, c))
        view.addLabel(os.path.basename(p),
                      {"fontSize": 11, "backgroundColor": "white",
                       "fontColor": "black", "inFront": True},
                      {"model": 0}, viewer=(r, c))
    view.zoomTo()
    view.show()
    print(f"백본 {n}개 비교 — 색상: 파랑(N말단) → 빨강(C말단) 잔기 index 그라디언트.")
    print("caption:", "생성물 = 컴퓨터가 낸 가설 구조입니다. 실제 발현·안정성은 실험 검증이 필요합니다.")


In [ ]:
# [강화] 백본별 구조 지표 — 실측 CA 좌표로 회전반경(Rg) 비교 (수치 날조 없음)
import os, glob
import numpy as np
import matplotlib.pyplot as plt
try:
    import seaborn as sns
    sns.set_theme(style="whitegrid", palette="colorblind")
except Exception:
    pass  # Colab 에는 기본 포함; 없으면 matplotlib 기본 스타일

def _read_ca_coords(pdb_path):
    xs = []
    with open(pdb_path) as f:
        for line in f:
            if line.startswith("ATOM") and line[12:16].strip() == "CA":
                xs.append((float(line[30:38]), float(line[38:46]), float(line[46:54])))
    return np.array(xs)

def _radius_of_gyration(coords):
    if len(coords) == 0:
        return np.nan
    c = coords - coords.mean(0)
    return float(np.sqrt((c ** 2).sum(1).mean()))

bb_pdbs = sorted(glob.glob("outputs/monomer_*.pdb"))
if not bb_pdbs:
    print("백본 PDB(outputs/monomer_*.pdb)가 없습니다 — 1단계(RFdiffusion) 실행 후 다시 시도하세요.")
else:
    names = [os.path.basename(p) for p in bb_pdbs]
    n_ca = [len(_read_ca_coords(p)) for p in bb_pdbs]
    rg = [_radius_of_gyration(_read_ca_coords(p)) for p in bb_pdbs]

    fig, axes = plt.subplots(1, 2, figsize=(10, 4), dpi=130)
    palette = plt.get_cmap("cividis")(np.linspace(0.15, 0.85, len(bb_pdbs)))
    axes[0].bar(names, n_ca, color=palette, edgecolor="black", linewidth=0.6)
    axes[0].set_title("백본별 잔기 수 (CA 원자 개수)")
    axes[0].set_ylabel("residue 수")
    axes[0].tick_params(axis="x", rotation=30)
    axes[1].bar(names, rg, color=palette, edgecolor="black", linewidth=0.6)
    axes[1].set_title("백본별 회전반경 Rg (컴팩트함 지표)")
    axes[1].set_ylabel("Rg (Å)")
    axes[1].tick_params(axis="x", rotation=30)
    fig.suptitle("생성 백본의 구조 다양성 — 실측 CA 좌표 기반", fontweight="bold")
    fig.text(0.5, -0.02,
             "생성물 = 가설. Rg 등 지표는 참고용이며 기능·안정성은 실험 검증 필요.",
             ha="center", fontsize=9, style="italic")
    fig.tight_layout()
    plt.show()


## 2. ProteinMPNN — 백본에 서열 설계 (inverse folding)

RFdiffusion 이 만든 백본에 대해, **그 구조로 접힐 만한 아미노산 서열**을 ProteinMPNN 으로 설계합니다.
하나의 백본에서 여러 후보 서열을 샘플링하며, `--sampling_temp` 로 다양성을 조절합니다(높을수록 다양·불안정).

> 근거(dauparas/ProteinMPNN README): `protein_mpnn_run.py --pdb_path ... --out_folder ... --num_seq_per_target ... --sampling_temp ...`
> ProteinMPNN 모델 가중치는 저장소에 포함되어 있어 별도 다운로드가 필요 없습니다.


In [ ]:
import os
# ProteinMPNN 저장소 clone (모델 가중치 포함)
if not os.path.isdir("ProteinMPNN"):
    os.system("git clone https://github.com/dauparas/ProteinMPNN.git")
print("ProteinMPNN 준비 완료:", os.path.isdir("ProteinMPNN"))


In [ ]:
import os
NUM_SEQS = 8          # 백본당 설계할 서열 후보 수
SAMPLING_TEMP = "0.2" # 샘플링 온도 (0.1~0.3 권장; 높을수록 다양)
os.makedirs("outputs/mpnn", exist_ok=True)

# ProteinMPNN 실행. 출처: dauparas/ProteinMPNN README
!python ProteinMPNN/protein_mpnn_run.py --pdb_path {backbone_pdb} --out_folder outputs/mpnn --num_seq_per_target {NUM_SEQS} --sampling_temp {SAMPLING_TEMP} --seed 37 --batch_size 1


In [ ]:
import os
# ProteinMPNN 출력 FASTA 위치: outputs/mpnn/seqs/<pdb이름>.fa
seq_dir = "outputs/mpnn/seqs"
fa_files = [os.path.join(seq_dir, f) for f in os.listdir(seq_dir)] if os.path.isdir(seq_dir) else []
print("생성된 FASTA:", fa_files)

designs = []  # (header, sequence) 목록
if fa_files:
    with open(fa_files[0]) as f:
        lines = [ln for ln in f.read().splitlines() if ln.strip()]
    for i in range(0, len(lines) - 1, 2):
        header, seq = lines[i], lines[i + 1]
        designs.append((header, seq))
    # ProteinMPNN 의 첫 레코드는 입력 백본 기준(native) 서열, 이후가 설계 샘플
    print(f"총 {len(designs)}개 레코드 (첫 레코드=입력 기준, 이후=설계 서열)\n")
    for header, seq in designs[:4]:
        print(header)
        print(seq)
        print("-" * 60)
else:
    print("FASTA 가 없습니다 — 2단계 실행 로그를 확인하세요.")


### 2-1. [강화] ProteinMPNN 점수 분포 & 설계 서열 다양성

ProteinMPNN 출력 FASTA 헤더에는 각 설계 서열의 **`score`(음의 로그가능도, 낮을수록 좋음)**,
**`global_score`**, **`seq_recovery`(입력 대비 동일 잔기 비율)**, 그리고 샘플링 **온도 `T`** 가 기록됩니다.
아래는 `outputs/mpnn/seqs/*.fa` 의 **실제 헤더 값만 파싱**해 분포를 그립니다 (합성/난수 없음, 파일 없으면 안내만 출력).

In [ ]:
# [강화] ProteinMPNN 점수 분포 — FASTA 헤더의 실제 값만 파싱 (score/global_score/seq_recovery/T)
import os, re, glob
import numpy as np
import matplotlib.pyplot as plt
try:
    import seaborn as sns
    sns.set_theme(style="whitegrid", palette="colorblind")
    _HAS_SNS = True
except Exception:
    _HAS_SNS = False

def parse_mpnn_fastas(seq_dir="outputs/mpnn/seqs"):
    """ProteinMPNN 출력 FASTA 들을 파싱. 첫 레코드=native(입력), 이후=설계 샘플."""
    recs = []
    if not os.path.isdir(seq_dir):
        return recs
    fa_files = sorted(glob.glob(os.path.join(seq_dir, "*.fa")) +
                      glob.glob(os.path.join(seq_dir, "*.fasta")))
    for fa in fa_files:
        try:
            lines = [ln for ln in open(fa).read().splitlines() if ln.strip()]
        except Exception:
            continue
        src = os.path.basename(fa)
        for j in range(0, len(lines) - 1, 2):
            h, seq = lines[j], lines[j + 1]
            if not h.startswith(">"):
                continue
            kv = dict(re.findall(r"([A-Za-z_]+)\s*=\s*([-\d.eE]+)", h))
            def g(k):
                try:
                    return float(kv[k])
                except (KeyError, ValueError):
                    return None
            is_native = ("sample" not in kv) and ("T" not in kv)
            s = seq.replace("/", "").replace(":", "").strip()
            recs.append({
                "source": src,
                "kind": "native" if is_native else "design",
                "T": g("T"), "sample": g("sample"),
                "score": g("score"), "global_score": g("global_score"),
                "seq_recovery": g("seq_recovery"),
                "seq": s, "length": len(s),
            })
    return recs

records = parse_mpnn_fastas()
designs_only = [r for r in records if r["kind"] == "design" and r["score"] is not None]

if not designs_only:
    print("파싱할 ProteinMPNN 설계 서열이 없습니다 — outputs/mpnn/seqs/*.fa 를 확인하세요 (2단계 실행 필요).")
else:
    scores = np.array([r["score"] for r in designs_only], dtype=float)
    gscores = np.array([r["global_score"] for r in designs_only
                        if r["global_score"] is not None], dtype=float)
    recov = np.array([r["seq_recovery"] for r in designs_only
                      if r["seq_recovery"] is not None], dtype=float)
    temps = sorted({r["T"] for r in designs_only if r["T"] is not None})

    fig, axes = plt.subplots(1, 3, figsize=(14, 4.2), dpi=130)

    # (1) score 히스토그램
    axes[0].hist(scores, bins=max(5, min(15, len(scores))),
                 color="#0072B2", edgecolor="black", alpha=0.85)
    axes[0].axvline(scores.mean(), color="#D55E00", ls="--", lw=1.5,
                    label=f"mean={scores.mean():.3f}")
    axes[0].set_title("설계 서열 score 분포\n(낮을수록 좋음)")
    axes[0].set_xlabel("score (neg. log-likelihood)")
    axes[0].set_ylabel("설계 수")
    axes[0].legend()

    # (2) 온도별 score 박스플롯 (온도 여러 개면 구분, 아니면 score vs global_score)
    if len(temps) > 1:
        data = [[r["score"] for r in designs_only if r["T"] == t] for t in temps]
        axes[1].boxplot(data, labels=[f"T={t}" for t in temps], showmeans=True)
        axes[1].set_title("샘플링 온도별 score")
        axes[1].set_xlabel("온도")
        axes[1].set_ylabel("score")
    else:
        box_data = [scores]
        labels = ["score"]
        if gscores.size:
            box_data.append(gscores)
            labels.append("global_score")
        axes[1].boxplot(box_data, labels=labels, showmeans=True)
        ttl = f"T={temps[0]}" if temps else "score / global_score"
        axes[1].set_title(f"score 요약 ({ttl})")
        axes[1].set_ylabel("값")

    # (3) seq_recovery
    if recov.size:
        axes[2].hist(recov * 100.0, bins=max(5, min(15, recov.size)),
                     color="#009E73", edgecolor="black", alpha=0.85)
        axes[2].axvline(recov.mean() * 100.0, color="#D55E00", ls="--", lw=1.5,
                        label=f"mean={recov.mean()*100:.1f}%")
        axes[2].set_title("입력 대비 서열 회복률\n(seq_recovery)")
        axes[2].set_xlabel("seq_recovery (%)")
        axes[2].set_ylabel("설계 수")
        axes[2].legend()
    else:
        axes[2].text(0.5, 0.5, "seq_recovery\n헤더 없음", ha="center", va="center")
        axes[2].set_axis_off()

    fig.suptitle(f"ProteinMPNN 설계 {len(designs_only)}개 — 실제 헤더 값 분포",
                 fontweight="bold")
    fig.text(0.5, -0.03,
             "생성물 = 가설. score/recovery 는 in silico 지표이며 실제 접힘·기능은 실험 검증 필요.",
             ha="center", fontsize=9, style="italic")
    fig.tight_layout()
    plt.show()


In [ ]:
# [강화] 설계 서열 다양성 — 실제 쌍별 sequence identity 히트맵 (동일 위치 잔기 비율)
import numpy as np
import matplotlib.pyplot as plt
try:
    import seaborn as sns
    _HAS_SNS = True
except Exception:
    _HAS_SNS = False

def pct_identity(a, b):
    n = min(len(a), len(b))
    if n == 0:
        return 0.0
    return 100.0 * sum(1 for i in range(n) if a[i] == b[i]) / n

_desql = [r for r in (designs_only if "designs_only" in dir() else []) if r.get("seq")]
if len(_desql) < 2:
    print("비교할 설계 서열이 2개 미만입니다 — 위 점수 분포 셀을 먼저 실행하거나 서열 수를 늘리세요.")
else:
    seqs = [r["seq"] for r in _desql]
    labels = [f"d{i}" + (f"(T{r['T']})" if r["T"] is not None else "")
              for i, r in enumerate(_desql)]
    N = len(seqs)
    M = np.array([[pct_identity(a, b) for b in seqs] for a in seqs])

    fig, ax = plt.subplots(figsize=(1.0 + 0.6 * N, 0.9 + 0.6 * N), dpi=130)
    annot = N <= 12
    if _HAS_SNS:
        sns.heatmap(M, xticklabels=labels, yticklabels=labels, cmap="viridis",
                    vmin=0, vmax=100, annot=annot, fmt=".0f",
                    cbar_kws={"label": "sequence identity (%)"}, ax=ax,
                    square=True, linewidths=0.5, linecolor="white")
    else:
        im = ax.imshow(M, cmap="viridis", vmin=0, vmax=100)
        ax.set_xticks(range(N)); ax.set_xticklabels(labels, rotation=45, ha="right")
        ax.set_yticks(range(N)); ax.set_yticklabels(labels)
        fig.colorbar(im, ax=ax, label="sequence identity (%)")
        if annot:
            for i in range(N):
                for j in range(N):
                    ax.text(j, i, f"{M[i, j]:.0f}", ha="center", va="center",
                            color="white" if M[i, j] < 50 else "black", fontsize=8)
    ax.set_title("설계 서열 쌍별 identity\n(낮을수록 다양)", fontweight="bold")
    fig.text(0.5, -0.04,
             "생성물 = 가설. 다양성 자체가 품질을 보장하지 않으며 실험 검증 필요.",
             ha="center", fontsize=9, style="italic")
    fig.tight_layout()
    plt.show()
    iu = np.triu_indices(N, k=1)
    if iu[0].size:
        print(f"쌍별 identity 평균 {M[iu].mean():.1f}% (범위 {M[iu].min():.0f}~{M[iu].max():.0f}%).")


In [ ]:
# [강화] sequence logo 용 logomaker 설치 (경량)
!pip install -q logomaker
print("logomaker 설치 시도 완료 (이미 있으면 그대로 사용).")


In [ ]:
# [강화] 위치별 아미노산 빈도 sequence logo — 실제 설계 서열에서 계산 (합성 없음)
import matplotlib.pyplot as plt

_desql = [r for r in (designs_only if "designs_only" in dir() else []) if r.get("seq")]
try:
    import logomaker
    _HAS_LOGO = True
except Exception as e:
    _HAS_LOGO = False
    print("logomaker 를 불러올 수 없습니다:", e, "— 위 설치 셀을 실행하세요.")

if _HAS_LOGO:
    if len(_desql) < 2:
        print("로고를 그릴 설계 서열이 2개 미만입니다 — 서열 수를 늘리거나 앞 셀을 먼저 실행하세요.")
    else:
        seqs = [r["seq"] for r in _desql]
        L = min(len(s) for s in seqs)
        seqs = [s[:L] for s in seqs]         # 동일 길이(단량체)로 정렬
        # 위치별 확률 행렬 = 실제 서열 빈도
        prob = logomaker.alignment_to_matrix(seqs, to_type="probability")
        show_L = min(L, 60)                    # 너무 길면 앞 60 위치만 표시
        fig, ax = plt.subplots(figsize=(min(16, 0.28 * show_L + 2), 3.2), dpi=130)
        logomaker.Logo(prob.iloc[:show_L], ax=ax, color_scheme="chemistry")
        ax.set_title(f"설계 서열 {len(seqs)}개의 위치별 아미노산 빈도 (앞 {show_L} 잔기)",
                     fontweight="bold")
        ax.set_xlabel("residue 위치")
        ax.set_ylabel("빈도 (probability)")
        fig.text(0.5, -0.06,
                 "생성물 = 가설. 위치별 보존 패턴은 참고용이며 기능은 실험 검증 필요.",
                 ha="center", fontsize=9, style="italic")
        fig.tight_layout()
        plt.show()
        if L > show_L:
            print(f"참고: 전체 {L} 잔기 중 앞 {show_L} 위치만 표시했습니다.")


## 3. (선택) Self-consistency 검증 — ESMFold 로 재접힘

**아이디어**: ProteinMPNN 이 설계한 서열을 **다시 3D 구조로 접어(fold)** 보고, 그 결과가 **RFdiffusion 이 원래 만든 백본과 얼마나 닮았는지** 봅니다.
많이 닮을수록(= self-consistent) "이 서열은 실제로 그 구조로 접힐 가능성이 높다"고 볼 수 있는, 설계 품질의 대표적 사전 스크리닝 지표입니다.

여기서는 **GPU 가 필요 없는 ESMFold 공개 API** 를 사용합니다.

> ESMFold API: `POST https://api.esmatlas.com/foldSequence/v1/pdb/` (본문에 아미노산 서열 문자열 → PDB 반환)
> 근거: Lin et al., Science 2023 (ESM Atlas). **주의**: 길이 제한(대략 400 residue 내외)과 서버 상태에 따라 실패할 수 있습니다. `# 확인 권장`


In [ ]:
import requests
# 설계 서열 하나 선택 (첫 sampled design = 두 번째 레코드; 없으면 첫 레코드)
design_seq = designs[1][1] if len(designs) > 1 else designs[0][1]
# ESMFold 는 표준 20종 아미노산만 받으므로 혹시 모를 구분자 제거
design_seq = design_seq.replace("/", "").replace(":", "").strip()
print("재접힘할 설계 서열 (len={}):".format(len(design_seq)))
print(design_seq)

# ESMFold API 호출 (GPU 불필요)
resp = requests.post("https://api.esmatlas.com/foldSequence/v1/pdb/",
                     data=design_seq, timeout=300)
refold_pdb = "outputs/esmfold_design.pdb"
if resp.status_code == 200 and resp.text.lstrip().startswith(("ATOM", "HEADER", "PARENT", "MODEL")):
    with open(refold_pdb, "w") as f:
        f.write(resp.text)
    print("\nESMFold 재접힘 성공 →", refold_pdb)
else:
    print("\nESMFold API 응답 실패 (상태코드 {}). 서버 상태/길이 제한 확인 권장.".format(resp.status_code))
    print(resp.text[:200])


In [ ]:
# 원 백본(회색)과 ESMFold 재접힘(청록)을 겹쳐 정성 비교
import py3Dmol
with open(backbone_pdb) as f:
    bb = f.read()
with open("outputs/esmfold_design.pdb") as f:
    rf = f.read()

view = py3Dmol.view(width=650, height=500)
view.addModel(bb, "pdb")
view.setStyle({"model": 0}, {"cartoon": {"color": "gray"}})
view.addModel(rf, "pdb")
view.setStyle({"model": 1}, {"cartoon": {"color": "cyan"}})
view.zoomTo()
view.show()
print("회색 = RFdiffusion 원 백본,  청록 = 설계 서열의 ESMFold 재접힘")
print("두 구조의 전체 fold 가 비슷할수록 self-consistent 한 설계입니다.")


In [ ]:
# (선택) 간이 CA-RMSD — 실측 좌표만 사용 (수치 날조 없음)
# 엄밀한 평가는 TM-align/US-align 를 쓰지만, 여기서는 개념 확인용 간이 지표만 계산합니다.
import numpy as np

def read_ca(pdb_path):
    coords = []
    with open(pdb_path) as f:
        for line in f:
            if line.startswith("ATOM") and line[12:16].strip() == "CA":
                coords.append((float(line[30:38]), float(line[38:46]), float(line[46:54])))
    return np.array(coords)

def kabsch_rmsd(P, Q):
    n = min(len(P), len(Q))          # 동일 순서로 매칭 (같은 길이 단량체)
    P, Q = P[:n], Q[:n]
    Pc, Qc = P - P.mean(0), Q - Q.mean(0)
    U, S, Vt = np.linalg.svd(Pc.T @ Qc)
    d = np.sign(np.linalg.det(Vt.T @ U.T))
    R = Vt.T @ np.diag([1, 1, d]) @ U.T
    Pr = Pc @ R.T
    return float(np.sqrt(((Pr - Qc) ** 2).sum() / n))

try:
    P, Q = read_ca(backbone_pdb), read_ca("outputs/esmfold_design.pdb")
    print(f"CA 개수 — 백본: {len(P)}, 재접힘: {len(Q)}")
    print(f"간이 CA-RMSD = {kabsch_rmsd(P, Q):.2f} Å")
    print("주의: 순서대로 매칭한 간이 지표입니다. 엄밀한 self-consistency 평가는 TM-align/US-align(TM-score) 권장.")
except Exception as e:
    print("RMSD 계산 생략:", e)


### 3-1. [강화] 설계별 self-consistency — 실측 CA-RMSD + ESMFold pLDDT

설계 서열 여러 개를 각각 ESMFold 로 재접힘해, **원 백본과의 CA-RMSD(실측 계산)** 와
**재접힘 구조의 평균 pLDDT(PDB B-factor 열에서 파싱)** 를 함께 봅니다.
RMSD 가 낮고 pLDDT 가 높을수록 self-consistent 한 설계입니다.
(공개 API 부하를 고려해 최대 몇 개만 재접힘하며, 이미 만든 PDB 는 재사용하고 없으면 안내만 출력합니다.)

In [ ]:
# [강화] 설계별 self-consistency: 다수 설계 재접힘 → 실측 CA-RMSD + 평균 pLDDT 막대차트
import os, shutil, requests
import numpy as np
import matplotlib.pyplot as plt
try:
    import seaborn as sns
    sns.set_theme(style="whitegrid", palette="colorblind")
except Exception:
    pass

N_SC = 4  # 재접힘할 설계 서열 최대 개수 (공개 API 부하 고려)
ESM_URL = "https://api.esmatlas.com/foldSequence/v1/pdb/"

def read_ca_bfac(pdb_path):
    coords, bf = [], []
    with open(pdb_path) as f:
        for line in f:
            if line.startswith("ATOM") and line[12:16].strip() == "CA":
                coords.append((float(line[30:38]), float(line[38:46]), float(line[46:54])))
                try:
                    bf.append(float(line[60:66]))
                except ValueError:
                    bf.append(np.nan)
    return np.array(coords), np.array(bf)

def sc_kabsch_rmsd(P, Q):
    n = min(len(P), len(Q))
    if n == 0:
        return np.nan
    P, Q = P[:n], Q[:n]
    Pc, Qc = P - P.mean(0), Q - Q.mean(0)
    U, S, Vt = np.linalg.svd(Pc.T @ Qc)
    d = np.sign(np.linalg.det(Vt.T @ U.T))
    R = Vt.T @ np.diag([1, 1, d]) @ U.T
    return float(np.sqrt((((Pc @ R.T) - Qc) ** 2).sum() / n))

_has_designs = ("designs" in dir()) and bool(designs)
_bb = backbone_pdb if "backbone_pdb" in dir() else ""
sc_results = []

if not _has_designs:
    print("designs 가 없습니다 — 2단계(ProteinMPNN) FASTA 파싱 셀을 먼저 실행하세요.")
elif not (_bb and os.path.isfile(_bb)):
    print("백본 PDB 가 없습니다(backbone_pdb) — 1단계 실행 후 다시 시도하세요.")
else:
    # 앞 셀에서 이미 접은 design(=designs[1]) 결과가 있으면 재사용
    if os.path.isfile("outputs/esmfold_design.pdb") and not os.path.isfile("outputs/esmfold_design_1.pdb"):
        try:
            shutil.copy("outputs/esmfold_design.pdb", "outputs/esmfold_design_1.pdb")
        except Exception:
            pass

    P_bb, _ = read_ca_bfac(_bb)
    sampled = [(i, s) for i, (h, s) in enumerate(designs) if i >= 1]  # 레코드 0 = native
    for idx, seq in sampled[:N_SC]:
        seq = seq.replace("/", "").replace(":", "").strip()
        out = f"outputs/esmfold_design_{idx}.pdb"
        if not os.path.isfile(out):
            try:
                resp = requests.post(ESM_URL, data=seq, timeout=300)
                if resp.status_code == 200 and resp.text.lstrip().startswith(
                        ("ATOM", "HEADER", "PARENT", "MODEL")):
                    with open(out, "w") as fh:
                        fh.write(resp.text)
                else:
                    print(f"design d{idx}: ESMFold 실패(status {resp.status_code}) — 건너뜀")
                    continue
            except Exception as e:
                print(f"design d{idx}: ESMFold 요청 오류({e}) — 건너뜀")
                continue
        Q, bf = read_ca_bfac(out)
        if len(Q) == 0:
            print(f"design d{idx}: 재접힘 PDB에 CA 없음 — 건너뜀")
            continue
        rmsd = sc_kabsch_rmsd(P_bb, Q)
        plddt = float(np.nanmean(bf)) if bf.size and not np.all(np.isnan(bf)) else np.nan
        sc_results.append({"design": f"d{idx}", "rmsd": rmsd, "plddt": plddt, "pdb": out})

    if not sc_results:
        print("재접힘 결과가 없습니다 — ESMFold API 상태/서열 길이(≈400 이하)를 확인하세요.")
    else:
        names = [r["design"] for r in sc_results]
        rmsds = [r["rmsd"] for r in sc_results]
        plddts = [r["plddt"] for r in sc_results]

        fig, ax1 = plt.subplots(figsize=(max(6, 1.4 * len(names)), 4.4), dpi=130)
        x = np.arange(len(names))
        b = ax1.bar(x - 0.2, rmsds, width=0.4, color="#0072B2",
                    edgecolor="black", linewidth=0.6, label="CA-RMSD (Å)")
        ax1.set_ylabel("CA-RMSD (Å)  ↓ 좋음", color="#0072B2")
        ax1.tick_params(axis="y", labelcolor="#0072B2")
        ax1.set_xticks(x); ax1.set_xticklabels(names)
        ax1.set_xlabel("설계 서열")
        for xi, v in zip(x - 0.2, rmsds):
            ax1.text(xi, v, f"{v:.2f}", ha="center", va="bottom", fontsize=8)

        ax2 = ax1.twinx()
        ax2.bar(x + 0.2, plddts, width=0.4, color="#E69F00",
                edgecolor="black", linewidth=0.6, label="mean pLDDT")
        ax2.set_ylabel("ESMFold 평균 pLDDT  ↑ 좋음", color="#E69F00")
        ax2.tick_params(axis="y", labelcolor="#E69F00")
        for xi, v in zip(x + 0.2, plddts):
            if not np.isnan(v):
                ax2.text(xi, v, f"{v:.0f}", ha="center", va="bottom", fontsize=8)

        lines1, lab1 = ax1.get_legend_handles_labels()
        lines2, lab2 = ax2.get_legend_handles_labels()
        ax1.legend(lines1 + lines2, lab1 + lab2, loc="upper right", fontsize=9)
        ax1.set_title("설계별 self-consistency (실측 CA-RMSD) + 재접힘 pLDDT",
                      fontweight="bold")
        fig.text(0.5, -0.03,
                 "생성물 = 가설. self-consistency 가 좋아도 실제 기능은 보장되지 않으며 실험 검증 필요.",
                 ha="center", fontsize=9, style="italic")
        fig.tight_layout()
        plt.show()
        best = min(sc_results, key=lambda r: r["rmsd"])
        print(f"가장 self-consistent 한 설계: {best['design']} "
              f"(CA-RMSD={best['rmsd']:.2f} Å, mean pLDDT={best['plddt']:.1f}).")
        print("주의: 간이 순서-매칭 CA-RMSD 입니다. 엄밀 평가는 TM-align/US-align(TM-score) 권장.")


In [ ]:
# [강화] 원 백본(회색) vs 최적 재접힘(청록) 겹침 뷰 — 뷰 내 범례 라벨 추가
import os
import py3Dmol

_bb = backbone_pdb if "backbone_pdb" in dir() else ""
# 최적 설계(sc_results 있으면 최저 RMSD) 아니면 기존 단일 재접힘 사용
if "sc_results" in dir() and sc_results:
    _best = min(sc_results, key=lambda r: r["rmsd"])
    _refold = _best["pdb"]
    _cap = f"{_best['design']} (CA-RMSD={_best['rmsd']:.2f} Å, pLDDT={_best['plddt']:.0f})"
else:
    _refold = "outputs/esmfold_design.pdb"
    _cap = "ESMFold 재접힘"

if not (_bb and os.path.isfile(_bb)):
    print("백본 PDB 가 없습니다 — 1단계 실행 후 다시 시도하세요.")
elif not os.path.isfile(_refold):
    print(f"재접힘 PDB({_refold})가 없습니다 — 위 self-consistency 셀을 먼저 실행하세요.")
else:
    with open(_bb) as f:
        bb = f.read()
    with open(_refold) as f:
        rf = f.read()
    view = py3Dmol.view(width=680, height=520)
    view.addModel(bb, "pdb")
    view.setStyle({"model": 0}, {"cartoon": {"color": "gray"}})
    view.addModel(rf, "pdb")
    view.setStyle({"model": 1}, {"cartoon": {"color": "cyan"}})
    # 뷰 내 범례
    view.addLabel("RFdiffusion 원 백본",
                  {"fontColor": "black", "backgroundColor": "lightgray",
                   "fontSize": 12, "inFront": True}, {"model": 0})
    view.addLabel(f"ESMFold 재접힘 — {_cap}",
                  {"fontColor": "white", "backgroundColor": "teal",
                   "fontSize": 12, "inFront": True}, {"model": 1})
    view.zoomTo()
    view.show()
    print("회색 = 원 백본,  청록 = 설계 서열의 ESMFold 재접힘. 겹칠수록 self-consistent.")
    print("caption:", "생성물 = 가설. 재접힘 일치는 사전 스크리닝 지표일 뿐 실험 검증이 필요합니다.")


## 4. 정리 & 정직한 한계

### 방금 한 것
1. **RFdiffusion** 으로 서열 없는 백본(100 residue) 을 확산 모델로 생성했습니다.
2. **ProteinMPNN** 으로 그 백본에 맞는 아미노산 서열 후보들을 설계했습니다.
3. (선택) **ESMFold** 로 설계 서열을 재접힘해 원 백본과 self-consistency 를 정성 비교했습니다.

### 반드시 알아야 할 한계
- **생성물 = 가설**: 여기서 나온 구조·서열은 *in silico* 예측일 뿐입니다. 실제 발현·용해도·안정성·기능은 **습식 실험(발현, SEC, CD, 결합/활성 assay 등)으로 검증**해야 합니다.
- **RFdiffusion 은 확률적(stochastic)**: 같은 설정이라도 매번 다른 백본이 나옵니다. 실무에서는 **다수 생성 → ProteinMPNN 다수 서열 → 재접힘 self-consistency(예: RMSD/TM-score, pLDDT) 로 필터링**하는 것이 표준입니다.
- **self-consistency ≠ 실제 기능**: 재접힘이 잘 맞아도 목적 기능(결합·촉매 등)을 보장하지 않습니다. 기능성 설계에는 hotspot·모티프 고정 등 조건부 설계와 추가 검증이 필요합니다.
- **성능 수치는 데이터로만**: 성공률·binding affinity 등은 이 노트북에서 만들어 말하지 않습니다. 반드시 자신의 실험/벤치마크 결과로 보고하세요.
- **재현성**: 시드(`--seed`), 모델 체크포인트, 각 도구 버전을 기록해야 결과를 재현할 수 있습니다.

### 다음 단계 (심화)
- 조건부 설계: 모티프 스캐폴딩, binder 설계(hotspot 지정), 대칭 올리고머 — ColabDesign `rf/examples` 의 다른 노트북 참고.
- 대량 스크리닝: 수백 개 백본 × 수 서열 → self-consistency 필터 → 상위 후보만 실험.


## 참고문헌 (실재 논문만)

1. **RFdiffusion** — Watson, J.L., Juergens, D., Bennett, N.R., et al. *De novo design of protein structure and function with RFdiffusion.* **Nature** 620, 1089–1100 (2023). https://doi.org/10.1038/s41586-023-06415-8
2. **ProteinMPNN** — Dauparas, J., Anishchenko, I., Bennett, N., et al. *Robust deep learning–based protein sequence design using ProteinMPNN.* **Science** 378, 49–56 (2022). https://doi.org/10.1126/science.add2187
3. **ESMFold** — Lin, Z., Akin, H., Rao, R., et al. *Evolutionary-scale prediction of atomic-level protein structure with a language model.* **Science** 379, 1123–1130 (2023). https://doi.org/10.1126/science.ade2574

### 도구/코드 저장소
- ColabDesign (설치·실행 레시피 출처): https://github.com/sokrypton/ColabDesign
- RFdiffusion (공식): https://github.com/RosettaCommons/RFdiffusion
- ProteinMPNN (공식): https://github.com/dauparas/ProteinMPNN
